<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/RAG_first_principles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Estimated run time:** ~5 minutes (embedding generation + LLM calls)


# RAG from First Principles — No Framework, No Magic


Code courtesy: This code has been adapted from towardsai.net/book code materials accompanying this book: [Building LLMs for production](https://www.oreilly.com/library/view/building-llms-for/9798324731472/)


In the previous chapter you may have seen RAG through a framework like LangChain, where a single function call hides all the plumbing. This notebook strips away every abstraction and builds the entire RAG pipeline by hand: chunk the text, embed the chunks, compute cosine similarity, retrieve the best matches, stuff them into a prompt, and call an LLM. By the end, you will understand every moving part well enough to build your own RAG system from scratch — or to debug one that someone else built with a framework.

The embedding step builds directly on the word-embedding ideas introduced in Chapter 2 — if the notion of mapping text to a vector space is still fuzzy, it is worth a quick detour back to the tokenization and embeddings demos before going further.

## Install and Import

We install only two things beyond standard Python: `openai` and `google-genai` for LLM access, `scikit-learn` for cosine similarity math, and `llm_cascade` — a lightweight wrapper that auto-detects whichever API key you have configured and routes requests to the cheapest available provider. There is no LangChain, no vector database library, no orchestration framework. Everything else we build ourselves.

In [ ]:
!pip install -qU git+https://github.com/KarAnalytics/llm_cascade.git openai google-genai scikit-learn


# Step 1: Load and Chunk the Dataset

Every RAG pipeline starts with raw text. Ours comes from a CSV of blog articles about the LLaMA2 model. The challenge is that these articles are too long to embed as single units — embedding models work best on passages of a few hundred to a thousand characters. So we need to **chunk** them: break each article into fixed-size pieces that are small enough to embed meaningfully, yet large enough to carry coherent information.

## Download the Data

We download two CSV files: one with the raw article text, and one with pre-computed embeddings (in case you want to skip the embedding step later). Both are hosted on GitHub so this notebook runs without any local file dependencies.

The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model. These articles will serve as our "knowledge base" — the documents that RAG will search through when answering questions.

## Chunk the Articles

The `split_into_chunks` function below is intentionally simple: it slices text into fixed 1,024-character windows with no overlap. Each article gets split independently, and all chunks are collected into a single list. We then load them into a Pandas DataFrame so we can attach embedding vectors as a new column later. This is the "R" foundation of RAG — without good chunks, retrieval cannot find the right context.

In [ ]:
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv

# Step 2: Generate Embeddings

Now we convert each text chunk into a numerical vector. An **embedding** is a list of numbers (in our case, 384 numbers) that captures the *meaning* of a piece of text. Two chunks about similar topics will have vectors that point in roughly the same direction; two chunks about unrelated topics will point in very different directions. This is the mathematical trick that makes semantic search possible — instead of matching keywords, we match meaning.

The embedding model we use (`all-MiniLM-L6-v2`) runs entirely on your machine. No API call, no cost, no data leaving your laptop. It is small and fast, but surprisingly effective for retrieval tasks.

In [ ]:
chunks = []

# Load the file as a CSV
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        chunks.extend(split_into_chunks(row[1]))

In [ ]:
# Convert the JSON list to a Pandas Dataframe
df = pd.DataFrame(chunks, columns=["chunk"])

df.keys()

After embedding, every row in our DataFrame now has two columns: the original text chunk and its 384-dimensional vector. This DataFrame *is* our vector database — just a table in memory. Production systems use dedicated vector stores (Pinecone, Weaviate, ChromaDB) for speed and persistence, but the underlying data structure is the same: text paired with its embedding.

# Step 3: Embed the User's Question

To find which chunks are relevant to a question, we need to put the question into the same vector space as the chunks. We embed the question using the exact same model (`all-MiniLM-L6-v2`), producing another 384-dimensional vector. Now we can compare apples to apples: the question vector lives in the same space as all the chunk vectors, and we can measure how "close" each chunk is to the question.

In [ ]:
# Embeddings use a local HuggingFace model (all-MiniLM-L6-v2)
# No API key needed — runs locally via llm.get_embedding()


# Step 4: Understand Cosine Similarity (A Quick Detour)

Before we search the full dataset, let us build intuition for how cosine similarity works. We will embed two sentences — one that clearly answers our question and one that is completely irrelevant — and compare their similarity scores to the question vector. If the math is working correctly, the relevant sentence should score close to 1.0 and the irrelevant one should score near 0.0.

Notice the scores: the irrelevant sentence ("The sky is blue") scores near zero, while the relevant sentence ("LLaMA2 model has a total of 2B parameters") scores close to 0.90. Cosine similarity ranges from -1 to 1, where 1 means identical direction in vector space. A score of 0.90 is very high — it tells us the embedding model "understands" that both the question and the good source are talking about the same topic. This is semantic search in action: no keyword matching, just meaning.

In [ ]:
# Define the user question, and convert it to embedding.
QUESTION = "How many parameters LLaMA2 model has?"
QUESTION_emb = llm.get_embedding(QUESTION)

len(QUESTION_emb)

# Step 5: Retrieve the Best Chunks

Now we apply cosine similarity at scale. We compare the question embedding against *every* chunk embedding in our DataFrame, producing a similarity score for each one. The chunks with the highest scores are the ones whose meaning is closest to the question — these are our retrieved context. We pick the top 3 and will feed them into the LLM prompt in the next step.

Calculating the similarity of embedding representations can help us to find pieces of text that are close to each other. In the following sample you see how the Cosine Similarity metric can identify which sentence could be a possible answer for the given user question. Obviously, the unrelated answer will score lower.


In [ ]:
BAD_SOURCE_emb = llm.get_embedding("The sky is blue.")
GOOD_SOURCE_emb = llm.get_embedding("LLaMA2 model has a total of 2B parameters.")

The indices `[52, 89, 114]` point to the three chunks whose embeddings are closest to our question. Let us inspect them to see if the retrieval makes sense — do they actually talk about LLaMA2 parameters? If they do, our embedding-based search is working. If they do not, we would need to revisit our chunking strategy or try a different embedding model.

## LLM Response Function

With the relevant chunks retrieved, we need a way to call an LLM. The `gemini_response` function below uses `llm_cascade` to send a prompt to whichever provider is available — Gemini, Ollama, Groq, OpenAI, or others. It accepts a system prompt (to set the LLM's role) and a user prompt (containing the question and context). This is a thin wrapper; the real work happens in how we *construct* the prompt in the next step.

# Step 6: Augment the Prompt and Generate

This is the "A" and "G" in RAG — **Augment** and **Generate**. We take the three retrieved chunks and insert them directly into the prompt between `<START_OF_CONTEXT>` and `<END_OF_CONTEXT>` tags. The system prompt tells the LLM to act as a context-grounded assistant that only answers AI-related questions. By embedding the retrieved text right in the prompt, we give the LLM specific evidence to draw from instead of relying on whatever it memorized during pre-training. The result should be a factual, grounded answer that cites information from the actual articles.

In [ ]:
number_of_chunks_to_retrieve = 3

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

## Bonus: How RAG Defeats Knowledge Cutoff and Hallucination

Every LLM has a **knowledge cutoff** — the date when its training data ended. Ask about events or models released after that date, and the LLM will either refuse to answer or, worse, confidently make something up (hallucinate). RAG solves this elegantly: by stuffing fresh, factual context into the prompt, we override the model's stale parametric memory with up-to-date evidence.

The next two cells demonstrate this with a concrete example. We provide a chunk about LLaMA 4 (released after most models' training cutoff) and ask the same question with and without that context. Watch how the augmented version gives a precise, correct answer while the non-augmented version either admits ignorance or hallucinates outdated information.

## LLM Response Function (via llm_cascade)

Instead of a hardcoded Gemini or OpenAI client, we use `llm.generate()` from `llm_cascade`. It automatically picks the first available provider and falls back if quota is exhausted. Each call prints which provider/model was used.


In [ ]:
# Using llm_cascade instead of hardcoded Gemini client

def gemini_response(system_prompt, prompt):
    """Generate a response using llm_cascade (auto-fallback across providers)."""
    try:
        response = llm.generate(prompt, system_prompt=system_prompt)
        return response.text
    except Exception as e:
        return f'Error: {e}'


# Without Augmentation — the Control Experiment

Now we ask the exact same question but *without* providing any context. The LLM must rely entirely on its parametric knowledge — whatever it memorized during training. If its training data predates the LLaMA 4 release, it will either say "I don't know" (the honest answer) or hallucinate incorrect details (the dangerous answer). Compare this output to the augmented version above to see exactly how much value the retrieval step adds.

## What We Built — and What Comes Next

Let us recap the full pipeline we built from scratch:

1. **Chunk** — Split raw articles into 1,024-character pieces.
2. **Embed** — Convert each chunk into a 384-dimensional vector using a local sentence-transformer.
3. **Store** — Keep the chunks and their embeddings in a simple Pandas DataFrame (our DIY vector database).
4. **Query** — Embed the user's question with the same model.
5. **Retrieve** — Use cosine similarity to find the top-k most relevant chunks.
6. **Augment** — Stuff those chunks into the LLM prompt as context.
7. **Generate** — Let the LLM produce a grounded answer.

Every RAG framework — LangChain, LlamaIndex, Haystack — automates these exact seven steps. Now that you have built them by hand, you understand what those frameworks are doing under the hood. In the next chapters, we will use LangChain to do this more concisely, and we will explore RAG beyond documents: with SQL databases and graph databases as knowledge sources.

In [ ]:
from google import genai
from google.genai import types

# Use the Gemini API to answer the questions based on the retrieved pieces of text.
try:
    # Formulating the system prompt and condition the model to answer only AI-related questions.
    system_prompt = """You are an assistant and expert in answering questions from a chunks of content.
                      Only answer AI-related question, else say that you cannot answer this question."""

    # User prompt with the user's question
    prompt = """
        Read the following informations that might contain the context you require to answer the question.
        You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag.
        Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
        Please provide an informative and accurate answer to the following question based on the avaiable context.
        Be concise and take your time. \nQuestion: {}\nAnswer:"
      """

    # Add the retrieved pieces of text to the prompt.
    prompt = prompt.format("".join(df.chunk[indices]), QUESTION)


    response = gemini_response(system_prompt,prompt)
    print(response)

except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
# Consider this as a retrieved chunk
# https://ai.meta.com/blog/llama-4-multimodal-intelligence/

Example_chunk = """
# Llama 4 Technical Overview

Meta has introduced the Llama 4 model family, marking a significant advancement in open-weight multimodal AI with three distinct variants designed for different use cases. The Llama 4 Scout represents the more compact option with 17 billion active parameters utilizing 16 experts within a total parameter count of 109 billion, designed to fit on a single H100 GPU with Int4 quantization. The Llama 4 Maverick serves as the flagship model with 17 billion active parameters but employs 128 experts across 400 billion total parameters, fitting on a single H100 host and designed as the primary workhorse for general assistant and chat applications. The preview model Llama 4 Behemoth functions as a teacher model with 288 billion active parameters, 16 experts, and nearly 2 trillion total parameters, currently still in training but demonstrating state-of-the-art performance on mathematical and reasoning benchmarks.
The architecture represents Meta's first implementation of mixture-of-experts (MoE) design in the Llama series, fundamentally changing how the models process information by activating only a fraction of total parameters for each token. The MoE architecture alternates between dense and mixture-of-experts layers, with the Maverick model specifically using 128 routed experts alongside a shared expert, where each token is processed by the shared expert and routed to one of the 128 specialized experts. This design significantly improves compute efficiency during both training and inference while delivering higher quality outputs compared to dense models with equivalent computational budgets.
Native multimodality represents another breakthrough, achieved through early fusion architecture that seamlessly integrates text and vision tokens into a unified model backbone during pre-training. This approach enables joint training with massive amounts of unlabeled text, image, and video data, with the vision encoder based on MetaCLIP but trained separately in conjunction with a frozen Llama model to optimize adaptation to the language model. The models support up to 48 images during pre-training and demonstrate strong performance with up to 8 images in post-training scenarios, enabling sophisticated visual reasoning and understanding tasks across multiple input modalities.
The training infrastructure incorporates several technical innovations, including the MetaP technique for reliable hyperparameter optimization that transfers well across different model configurations and training scenarios. The models were trained on over 30 trillion tokens, representing more than double the training data of Llama 3, with support for 200 languages including over 100 languages with more than 1 billion tokens each. Training efficiency was maximized through FP8 precision without quality sacrifice, achieving 390 TFLOPs per GPU utilization on 32,000 GPUs during Behemoth pre-training, while a specialized mid-training phase enhanced core capabilities and enabled the industry-leading 10 million token context length for Scout.
Context length capabilities represent a dramatic advancement, with Llama 4 Scout supporting 10 million tokens compared to Llama 3's 128K limit through the innovative iRoPE architecture that employs interleaved attention layers without positional embeddings and inference-time temperature scaling of attention. This architecture enables superior length generalization and opens possibilities for multi-document summarization, extensive user activity parsing for personalization, and reasoning over vast codebases, with compelling performance demonstrated in retrieval tasks and cumulative negative log-likelihood evaluations over 10 million tokens of code.
The post-training pipeline underwent significant revision with a three-stage approach consisting of lightweight supervised fine-tuning, online reinforcement learning, and lightweight direct preference optimization. A critical insight involved removing over 50% of data tagged as easy using Llama models as judges, focusing training on harder examples to prevent over-constraining that could restrict exploration during online RL and lead to suboptimal performance in reasoning, coding, and mathematics. The continuous online RL strategy alternated between model training and adaptive data filtering to retain only medium-to-hard difficulty prompts, proving highly beneficial for compute and accuracy trade-offs.
Safety and bias mitigation received comprehensive attention through multi-layered approaches spanning pre-training data filtering, post-training policy conformance, and system-level safeguards. Meta introduced Generative Offensive Agent Testing (GOAT) to address traditional red-teaming limitations by simulating multi-turn interactions of medium-skilled adversarial actors, enabling more efficient vulnerability detection while allowing human experts to focus on novel adversarial areas. Particular emphasis was placed on addressing political and social bias inherent in internet training data, with Llama 4 demonstrating significant improvements over Llama 3 in presenting balanced perspectives on contentious issues without favoring particular viewpoints.
The distillation process for creating smaller models from Behemoth required novel approaches, including a dynamic loss function that weighted soft and hard targets throughout training, with codistillation during pre-training amortizing computational costs for the majority of training data. Post-training the 2 trillion parameter Behemoth model necessitated pruning 95% of supervised fine-tuning data compared to 50% for smaller models, followed by large-scale reinforcement learning focused on sampling hard prompts through pass@k analysis and constructing training curricula of increasing difficulty. The unprecedented scale required complete infrastructure overhaul, including optimized MoE parallelization and fully asynchronous online RL training framework that improved training efficiency by approximately 10x over previous generations through flexible GPU allocation and resource balancing across multiple models.
"""

In [ ]:
QUESTION = "How many parameters LLaMA 4 model has?"

# Formulating the system prompt
system_prompt = """ You are an assistant and expert in answering questions from a chunks of content.
                    Only answer AI-related question, else say that you cannot answer this question."""

# Combining the system prompt with the user's question
prompt = """Read the following informations that might contain the context you require to answer the question.
            You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag.
            Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n".
            Please provide an informative and accurate answer to the following question based on the avaiable context.
            Be concise and take your time. \nQuestion: {}\nAnswer:
            """

prompt = prompt.format(Example_chunk, QUESTION)

response = gemini_response(system_prompt,prompt)
print(response)

In [ ]:
QUESTION = "How many parameters LLaMA 4 model has?"

# Formulating the system prompt
system_prompt = "You are an assistant and expert in answering questions."

# Combining the system prompt with the user's question
prompt = "Be concise and take your time to answer the following question. \nQuestion: {}\nAnswer:"

prompt = prompt.format(QUESTION)

response = gemini_response(system_prompt,prompt)
print(response)

## Key takeaways

- **RAG is seven concrete steps**: chunk, embed, store, query-embed, retrieve, augment, generate -- frameworks hide them, but each is plain Python.
- **Cosine similarity on sentence-transformer embeddings** turns semantic search into a matrix operation: relevant chunks score near 1, unrelated ones near 0.
- **A Pandas DataFrame is a valid vector store** for learning and small datasets; production systems just swap in Pinecone, Weaviate, or ChromaDB for speed and persistence.
- **Augmenting the prompt with retrieved chunks** grounds the LLM in fresh facts, defeating both knowledge-cutoff gaps and hallucination in a single shot.
- **Local embedding models** like `all-MiniLM-L6-v2` keep the "R" in RAG free and private, while `llm_cascade` handles provider fallback for the "G".